In [353]:
# Brian Loch (Refactored)
# Packages
import pandas as pd
import random
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

In [393]:
# --- Global Configuration ---
indep_vars = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed', 'height_m', 'weight_kg']
Seed = 5596

# --- Load and Prepare Data ---
df = pd.read_csv('../data/pokemon_complete.csv')
X = df[indep_vars]

# Prepare Multi-Label Target (Types 1 & 2)
df['all_types'] = df[['type_1', 'type_2']].apply(
    lambda x: [t for t in x if pd.notna(t)], axis=1
)
mlb = MultiLabelBinarizer()
y_multi = mlb.fit_transform(df['all_types'])

# Prepare Single-Label Target (Type 1 only)
le = LabelEncoder()
y_single = le.fit_transform(df['type_1'])

# --- Accuracy Check ---
# Using consistent test_size for comparison, 
# best accuracy (66.67%) so far have been with 
# Seed = 5596, test_size = 0.011 and n_estimators= 750+
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y_single, test_size=0.011, random_state=Seed, shuffle=True
)

model_multi = RFC(n_estimators=750, random_state=Seed)
model_multi.fit(X_train_s, y_train_s)

accuracy = accuracy_score(y_test_s, model_multi.predict(X_test_s))
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 66.67%


In [355]:
# --- Prediction Functions ---

def test_pokemon_by_row(pkmn_row, threshold=0.20):
    sample = pd.DataFrame([pkmn_row[indep_vars].values], columns=indep_vars)
    
    # Get probabilities
    probs = model_multi.predict_proba(sample)[0]
    
    # Pair the type names with their calculated probabilities
    type_probabilities = list(zip(le.classes_, probs))
    
    # Sort the list by probability in descending order
    type_probabilities.sort(key=lambda x: x[1], reverse=True)
    
    # Apply the threshold AND limit to the top 2 results
    predicted_types = [
        name for name, prob in type_probabilities[:2] 
        if prob >= threshold
    ]
    
    actual_types = pkmn_row['all_types']

    print("-" * 40)
    print(f"POKEMON: {pkmn_row['name']} (No. {pkmn_row['pokedex_number']})")
    print("-" * 40)
    print(f"ACTUAL TYPES   : {actual_types}")
    print(f"PREDICTED TYPES: {predicted_types}")
    print("-" * 40)

def test_random_pokemon(threshold=0.20):
    random_idx = random.randint(0, len(df) - 1)
    test_pokemon_by_row(df.iloc[random_idx], threshold)

In [389]:
# --- Run Test ---
test_random_pokemon()

----------------------------------------
POKEMON: Oinkologne-Female (No. 10254)
----------------------------------------
ACTUAL TYPES   : ['Normal']
PREDICTED TYPES: ['Normal']
----------------------------------------
